# 03 - Visualize Stuff-Things Classification

Visualization of the stuff-things classifier and its input cues.

**Goals:**
- Visualize DBD, FCC, IDF cues per cluster
- Show stuff vs things classification results
- Compare with ground truth stuff/things labels
- Analyze classifier confidence distribution

In [ ]:
import sys
sys.path.insert(0, '..')

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

rng = jax.random.PRNGKey(0)

## 1. Generate Synthetic Cues

In [ ]:
from mbps.models.classifier.cues import (
    compute_depth_boundary_density,
    compute_feature_cluster_compactness,
    compute_instance_decomposition_freq,
)

# Simulate cluster assignments and features
num_clusters = 27
num_pixels = 784  # 28x28

clusters = jax.random.randint(rng, (num_pixels,), 0, num_clusters)
depth = jax.random.uniform(rng, (28, 28)) * 10.0
dino_features = jax.random.normal(rng, (num_pixels, 384))
instance_masks = jax.nn.sigmoid(jax.random.normal(rng, (50, num_pixels)))

print(f'Clusters: {clusters.shape}, unique: {len(np.unique(np.array(clusters)))}')
print(f'Depth: {depth.shape}')
print(f'Features: {dino_features.shape}')
print(f'Instance masks: {instance_masks.shape}')

## 2. Stuff-Things MLP Classifier

In [ ]:
from mbps.models.classifier.stuff_things_mlp import StuffThingsClassifier

# Create synthetic cues [DBD, FCC, IDF]
cues = jax.random.normal(rng, (1, num_clusters, 3))

classifier = StuffThingsClassifier()
cls_params = classifier.init(rng, cues)
scores = classifier.apply(cls_params, cues)[0]  # (K,)

print(f'Stuff-Things scores range: [{float(scores.min()):.3f}, {float(scores.max()):.3f}]')
print(f'Thing clusters (score > 0.5): {int(jnp.sum(scores > 0.5))}')
print(f'Stuff clusters (score <= 0.5): {int(jnp.sum(scores <= 0.5))}')

## 3. Visualize Score Distribution

In [ ]:
scores_np = np.array(scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(scores_np, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(x=0.5, color='red', linestyle='--', label='Threshold')
axes[0].set_xlabel('Thing Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Stuff-Things Score Distribution')
axes[0].legend()

# Bar plot per cluster
colors = ['tab:blue' if s <= 0.5 else 'tab:orange' for s in scores_np]
axes[1].bar(range(num_clusters), scores_np, color=colors, alpha=0.8)
axes[1].axhline(y=0.5, color='red', linestyle='--')
axes[1].set_xlabel('Cluster ID')
axes[1].set_ylabel('Thing Score')
axes[1].set_title('Per-Cluster Classification')

plt.tight_layout()
plt.show()

## 4. Visualize Cues Correlation

In [ ]:
cues_np = np.array(cues[0])  # (K, 3)
cue_names = ['DBD', 'FCC', 'IDF']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (ax, name) in enumerate(zip(axes, cue_names)):
    scatter = ax.scatter(
        cues_np[:, i], scores_np,
        c=scores_np, cmap='RdYlBu_r', alpha=0.7, edgecolors='black', s=40
    )
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel(name)
    ax.set_ylabel('Thing Score')
    ax.set_title(f'{name} vs Thing Score')

plt.colorbar(scatter, ax=axes[-1], label='Score')
plt.tight_layout()
plt.show()

## Summary

Stuff-things classifier analysis:
- Classifier produces scores in [0, 1] for each cluster
- Threshold at 0.5 separates stuff from things
- Cue correlations show which features drive classification

Next: `04_analyze_results.ipynb`